# Task 1 — Merging Customer Social Profiles and Transactions

In this notebook, I explore and clean the two tabular datasets, then join customer social-profile information to transaction records. The merged file can be used as input for the product recommendation work in a later task; this notebook does not train a model.

## Before running

1. Open the [customer social profiles spreadsheet](https://docs.google.com/spreadsheets/d/10up-WdC0a6egYaXLKiMQUotpOvaKZRZvYuWFQx4-RPQ/edit?gid=862127784#gid=862127784) and export the relevant sheet as a CSV named `customer_social_profiles.csv`.
2. Open the [customer transactions spreadsheet](https://docs.google.com/spreadsheets/d/1s4WOVm49lmLQ8d9QbbbdgAcRTNh3m0KCH_5ciiaRZw0/edit?gid=1844409263#gid=1844409263) and export the relevant sheet as a CSV named `customer_transactions.csv`.
3. Put both files in a `data` folder next to this notebook, then run the cells in order. The spreadsheet links may require your course account.

The join key is selected from columns present in both files. The notebook only auto-suggests a key when there is exactly one familiar customer-ID column; otherwise, I set it explicitly after checking the column names.

## 1. Import libraries and set file paths

I use pandas for tabular data cleaning and merging, and matplotlib/seaborn for exploratory plots. If a library is missing, install it in the notebook environment with `pip install pandas numpy matplotlib seaborn`.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_DIR = Path("data")
SOCIAL_PATH = DATA_DIR / "customer_social_profiles.csv"
TRANSACTIONS_PATH = DATA_DIR / "customer_transactions.csv"
OUTPUT_PATH = DATA_DIR / "merged_customer_transactions_social.csv"

print(f"Social profiles file: {SOCIAL_PATH.resolve()}")
print(f"Transactions file: {TRANSACTIONS_PATH.resolve()}")

## 2. Load the CSV files and standardize column names

Standardizing headers makes later column comparisons easier. I keep the data values unchanged here and check for duplicate column names after normalization.

In [ ]:
def normalize_column_name(column):
    """Convert a column label to lowercase snake_case for consistent matching."""
    name = str(column).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def load_and_standardize_csv(path):
    """Read a CSV and normalize its headers, raising a useful error if absent."""
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Export the spreadsheet as CSV and place it there."
        )
    frame = pd.read_csv(path)
    frame.columns = [normalize_column_name(column) for column in frame.columns]
    if frame.columns.duplicated().any():
        duplicates = frame.columns[frame.columns.duplicated()].tolist()
        raise ValueError(f"Header normalization created duplicate column names: {duplicates}")
    return frame


social = load_and_standardize_csv(SOCIAL_PATH)
transactions = load_and_standardize_csv(TRANSACTIONS_PATH)

print("Social profiles shape:", social.shape)
print("Transactions shape:", transactions.shape)

## 3. Initial data checks

I check a few rows, data types, descriptive statistics, missing values, exact duplicate rows, and columns shared by the two sources. These checks help me understand the actual files before choosing the join key.

In [ ]:
for table_name, frame in [("Social profiles", social), ("Transactions", transactions)]:
    print(f"\n--- {table_name}: sample rows ---")
    display(frame.head())
    print(f"\n--- {table_name}: data types ---")
    display(frame.dtypes.rename("dtype").to_frame())
    print(f"\n--- {table_name}: descriptive statistics ---")
    display(frame.describe(include="all").T)
    print(f"\n--- {table_name}: missing values ---")
    display(frame.isna().sum().rename("missing_count").to_frame())
    print(f"Exact duplicate rows: {frame.duplicated().sum()}")

common_columns = sorted(set(social.columns).intersection(transactions.columns))
print("\nColumns found in both files:", common_columns)

## 4. Exploratory plots

The plots below are generated from numeric columns that actually exist in the CSVs. Their shapes and any visible outliers should be interpreted using the dataset's units and context; I do not assume any particular pattern before seeing the data.

### Plot 1 — Numeric feature distributions

Histograms show the spread and skew of numeric fields. A long tail or multiple peaks may be worth checking, but the plot alone does not explain why those values occur.

In [ ]:
numeric_fields = []
for table_name, frame in [("transactions", transactions), ("social", social)]:
    for column in frame.select_dtypes(include=np.number).columns:
        numeric_fields.append((table_name, column, frame[column].dropna()))

if numeric_fields:
    plot_fields = numeric_fields[: min(4, len(numeric_fields))]
    fig, axes = plt.subplots(1, len(plot_fields), figsize=(5 * len(plot_fields), 4))
    axes = np.atleast_1d(axes)
    for axis, (table_name, column, values) in zip(axes, plot_fields):
        sns.histplot(values, bins=20, kde=len(values) > 1, ax=axis)
        axis.set(title=f"{table_name}: {column}", xlabel=column, ylabel="Row count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns were detected, so a numeric distribution plot is not applicable.")

### Plot 2 — Numeric feature boxplots

Boxplots make extreme values easier to spot relative to the rest of each feature. Potential outliers should be investigated rather than removed automatically, because they may be valid purchases or customer values.

In [ ]:
if numeric_fields:
    plot_fields = numeric_fields[: min(4, len(numeric_fields))]
    fig, axes = plt.subplots(1, len(plot_fields), figsize=(5 * len(plot_fields), 4))
    axes = np.atleast_1d(axes)
    for axis, (table_name, column, values) in zip(axes, plot_fields):
        sns.boxplot(y=values, ax=axis)
        axis.set(title=f"{table_name}: {column}", xlabel="", ylabel=column)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns were detected, so a numeric boxplot is not applicable.")

### Plot 3 — Numeric feature correlations

The heatmap shows pairwise correlations among numeric columns within one source table. Correlation is not causation, and fields from separate tables are not combined here because their row order may not represent the same customers.

In [ ]:
correlation_source_name = None
correlation_source = None
for table_name, frame in [("transactions", transactions), ("social profiles", social)]:
    numeric_frame = frame.select_dtypes(include=np.number)
    if numeric_frame.shape[1] >= 2:
        correlation_source_name = table_name
        correlation_source = numeric_frame
        break

if correlation_source is not None:
    correlation = correlation_source.corr()
    plt.figure(figsize=(max(6, 1.2 * len(correlation.columns)), max(4, 1.0 * len(correlation.columns))))
    sns.heatmap(correlation, annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title(f"Numeric feature correlations — {correlation_source_name}")
    plt.tight_layout()
    plt.show()
else:
    print("A correlation heatmap needs at least two numeric columns in the same source; none were available.")

## 5. Remove exact duplicate rows

I remove only rows that are exact duplicates across every column and report how many were removed. I do not remove rows just because one field is repeated; repeated transaction customers are expected in a transaction table.

In [ ]:
def drop_exact_duplicates(frame, table_name):
    """Drop fully identical rows and report how many rows were removed."""
    duplicate_count = int(frame.duplicated().sum())
    cleaned = frame.drop_duplicates().copy()
    print(f"{table_name}: removed {duplicate_count} exact duplicate row(s).")
    return cleaned


social = drop_exact_duplicates(social, "Social profiles")
transactions = drop_exact_duplicates(transactions, "Transactions")

## 6. Choose and check the merge key

I merge on a customer identifier shared by both files. If the automatic suggestion is missing or ambiguous, I inspect `common_columns` above and enter the correct normalized column name in `MERGE_KEY` before rerunning this cell.

In [ ]:
# Set this to a shared column name if automatic detection is missing or ambiguous.
MERGE_KEY = None

candidate_names = {"customer_id", "customerid", "user_id", "userid", "client_id", "clientid"}
shared_candidates = [column for column in common_columns if column in candidate_names]

if MERGE_KEY is None:
    if len(shared_candidates) != 1:
        raise ValueError(
            "Could not identify exactly one customer ID column automatically. "
            f"Shared columns: {common_columns}. Set MERGE_KEY to the correct shared column and rerun."
        )
    MERGE_KEY = shared_candidates[0]
else:
    MERGE_KEY = normalize_column_name(MERGE_KEY)

if MERGE_KEY not in social.columns or MERGE_KEY not in transactions.columns:
    raise KeyError(
        f"MERGE_KEY={MERGE_KEY!r} must be present in both tables. "
        f"Social columns: {social.columns.tolist()}; transaction columns: {transactions.columns.tolist()}"
    )

print("Selected merge key:", MERGE_KEY)
print("Social-profile null keys:", int(social[MERGE_KEY].isna().sum()))
print("Transaction null keys:", int(transactions[MERGE_KEY].isna().sum()))
print("Duplicate social-profile keys:", int(social[MERGE_KEY].duplicated(keep=False).sum()))
print("Repeated transaction keys (usually expected):", int(transactions[MERGE_KEY].duplicated(keep=False).sum()))

## 7. Validate and merge the two tables

I use transactions as the left table so every purchase row (including the product field for a later task) is retained. The merge expects one social-profile row per customer; if multiple different profile rows remain for a customer, I stop instead of multiplying transaction rows. Profile rows with missing IDs are excluded from matching, while transactions with missing IDs stay in the output as unmatched records.

In [ ]:
profile_duplicate_mask = social[MERGE_KEY].notna() & social[MERGE_KEY].duplicated(keep=False)
if profile_duplicate_mask.any():
    duplicate_profile_keys = social.loc[profile_duplicate_mask, MERGE_KEY].drop_duplicates().head(10).tolist()
    raise ValueError(
        "The social-profile table has multiple rows for a customer after exact duplicate removal. "
        f"Example keys: {duplicate_profile_keys}. Review these records and choose a deliberate resolution "
        "before merging; automatic aggregation could change feature meaning."
    )

# Remove missing lookup keys so pandas cannot match a missing ID to another missing ID.
profiles_for_merge = social.loc[social[MERGE_KEY].notna()].copy()
print("Profile rows excluded because the merge key is missing:", len(social) - len(profiles_for_merge))

transaction_row_count = len(transactions)
merged = transactions.merge(
    profiles_for_merge,
    on=MERGE_KEY,
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("_transaction", "_social"),
)

if len(merged) != transaction_row_count:
    raise AssertionError(
        f"Expected {transaction_row_count} transaction rows after a left merge; got {len(merged)}."
    )

match_counts = merged["_merge"].value_counts()
matched_count = int(match_counts.get("both", 0))
unmatched_count = int(match_counts.get("left_only", 0))
match_rate = matched_count / len(merged) if len(merged) else 0.0

print("Transaction rows before merge:", transaction_row_count)
print("Rows after merge:", len(merged))
print("Matched transaction rows:", matched_count)
print("Unmatched transaction rows:", unmatched_count)
print(f"Transaction match rate: {match_rate:.1%}")
display(match_counts.rename("row_count").to_frame())

## 8. Review and save the merged dataset

I review the joined rows and missing-value counts, then save the result as a CSV. The `_merge` indicator is kept so matched and unmatched transactions can be checked later.

In [ ]:
print("Merged dataset shape:", merged.shape)
print("Merged columns:", merged.columns.tolist())
display(merged.head())

missing_after_merge = merged.isna().sum().sort_values(ascending=False)
display(missing_after_merge[missing_after_merge > 0].rename("missing_count").to_frame())

DATA_DIR.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_PATH, index=False)
print(f"Saved merged dataset to: {OUTPUT_PATH.resolve()}")

## 9. Merge validation checklist and takeaways

- The selected key exists in both source tables; review its meaning and type before relying on the result.
- Exact duplicate rows were removed, and conflicting duplicate profile IDs cause the merge to stop for manual review.
- A left join preserved the transaction row count; the match counts and missing-value summary show merge coverage.
- The output file is `data/merged_customer_transactions_social.csv`. Check `_merge` and the target product column before using the data in a later recommendation task.

A low match rate can indicate inconsistent customer IDs, different ID formatting, or customers with no social-profile row. I would investigate those cases rather than assume the join is correct.